# SigAlg's `integrate` method

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `Operators.integrate` method in SigAlg is a method for computing *Lebesgue integrals* of random variables and vectors. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.Operators.integrate).

## Mathematical definition

Let $X: \Omega \to \mathbb{R}$ be a random variable on a probability space $(\Omega, \mathcal{F},P)$. This method computes the Lebesgue integral

$$
\int_A X \, dP,
$$

where $A$ is an event in $\mathcal{F}$. If $\Omega$ is finite (as it always is, in SigAlg), then the Lebesgue integral reduces to a finite sum

$$
\sum_{\omega \in A} X(\omega) P(\{\omega\}).
$$

While in the mathematical theory $A$ is supposed to be an $\mathcal{F}$-measurable subset of $\Omega$, this requirement is not enforced in SigAlg. If the event $A$ is not specified, it defaults to the sample space itself $A = \Omega$. If the measure $P$ is not specified, it defaults to the measure carried by the random variable in its `probability_measure` attribute.

If $X:\Omega \to \mathbb{R}^d$ is instead a random vector of dimension $d>1$, with components

$$
X = (X_1, X_2, \ldots, X_d),
$$

then this method returns a `pd.Series` object whose values are the separate Lebesgue integrals $\int_A X_j \, dP$, for $j=1,2,\ldots,d$.

## API examples

### Integrals of random variables

First define a sample space $\Omega = \{0,1,2,3\}$ and a probability measure $P$ on $\Omega$:

In [10]:
from sigalg.core import ProbabilityMeasure, SampleSpace

Omega = SampleSpace().from_sequence(size=4)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.4,
        2: 0.2,
        3: 0.3
    }
)

print(Omega, "\n")
print(P)

Sample space 'Omega':
[0, 1, 2, 3] 

Probability measure 'P':
        probability
sample             
0               0.1
1               0.4
2               0.2
3               0.3


Define a random variable $X: \Omega \to \mathbb{R}$ and set its `probability_measure` attribute to $P$ so that the Lebesgue integral will be computed against $P$. Then, compute the Lebesgue integral $\int_\Omega X \, dP$:

In [11]:
from sigalg.core import Operators, RandomVariable

X = RandomVariable(domain=Omega).from_dict(
    {
        0: -2,
        1: 1,
        2: 4,
        3: -1,
    }
)
X.probability_measure = P

int = Operators.integrate

print(X, "\n")
print(int(X))

Random variable 'X':
        X
sample   
0      -2
1       1
2       4
3      -1 

0.7


Now integrate over the event $A = \{0,1\}$:

In [12]:
A = Omega.get_event([0, 1])

print(int(X, event=A))

0.2


### Integrals of random vectors

We now define a $3$-dimensional random vector $Y:\Omega \to \mathbb{R}^3$ and compute its Lebesgue integral:

In [13]:
from sigalg.core import RandomVector

Y = RandomVector(domain=Omega, name="Y").from_dict(
    {
        0: (1, -2, 3),
        1: (-2, 3, 4),
        2: (-3, 0, 0),
        3: (9, -2, 4),
    }
)
Y.probability_measure = P

print(int(Y))

integral
integral(Y_0)    1.4
integral(Y_1)    0.4
integral(Y_2)    3.1
Name: integral(Y), dtype: float64


Compute the integrals of the components of $Y$ individually, and check that they match the previous computation:

In [14]:
for i, component in enumerate(Y.components):
    print(f"The integral of the {i}-th component of 'Y': {round(int(component),2)}")

The integral of the 0-th component of 'Y': 1.4
The integral of the 1-th component of 'Y': 0.4
The integral of the 2-th component of 'Y': 3.1


### Testing properties of integrals

#### The integral is a finite sum

We noted in the definition that the integral is given by the finite sum $\int_\Omega X \, dP = \sum_{\omega \in \Omega} X(\omega)P(\{\omega\})$. In the next cell, we check this equality.

In [15]:
int_as_sum = sum([X(omega) * P(omega) for omega in Omega])

print(int(X), "\n")
print(int_as_sum)

0.7 

0.7
